In [1]:
import spacy
from FlagEmbedding import FlagModel
from elasticsearch import Elasticsearch
from elasticsearch import helpers

from tqdm import tqdm

In [62]:
# IndexText uses an embedding model to generate vector embeddings of an input text and stores these embeddings in a 
# vector database using Elasticsearch. The input text in split into chunks of multiple sentences, then embedded and indexed.
# The vector database takes the form (text_chunk_i, vector_embedding_i) for the ith entry of the index.

In [2]:
# helper functions
def remove_newline(text):
# removes newline characters, "\n", from text
# text: list of paragraphs in the text
    
    for i in range(len(text)):
        text[i] = " ".join(text[i].split())

    return text

def embed_index_text(text_chunks, client):
    # chunks: list of m elements that contain n sentences each
    # client: instance of Elasticsearch client used to create the index

    # embed chunks
    chunk_embeddings = model.encode(text_chunks).tolist()

    # define the format of the data to be indexed as pairs (chunk of text, chunk embeddings)
    docs = [
        {
            '_op_type': 'index',
            '_index': 'les_miserables_index',
            '_source': {
                "chunk" : t, 
                "embedding_vector" : v
            }
        } for t, v in zip(text_chunks, chunk_embeddings)
    ]
    
    # index in bulk
    res = helpers.bulk(client, docs)
    # print(res)

def createIndex(client, index_name):
    # client: instance of Elasticsearch client
    # index_name (str): index name

    # ensure that there is no previously defined index under the index_name
    if (client.indices.exists(index = index_name)):
        client.indices.delete(index = index_name)

    # define the format of index: chunk of text and embedding vectors
    # custom mapping that defines the expected types of indices features
    # define mapping parameters for the "chunk" and "embedding_vector" fields
    # define "vector_dim"
    mappings = {
        "properties": {
            "chunk": {
                "type": "text"
            }, 
            "embedding_vector": {
                "index": True, 
                "type": "dense_vector", 
                "dims": 512, 
                "similarity": "cosine",
            }
        }
    }
        
    # create index
    client.indices.create(index = index_name, mappings = mappings)


In [3]:
class IndexText:

    def __init__(self, embedding_model, client):
        self.embedding_model = FlagModel('BAAI/bge-small-zh-v1.5', use_fp16 = True)
        self.client = client

    def __preProcessInput(self, file_path):
        # text: text file

        with(open(file_path, "r")) as text_file:
            text = text_file.read()

        # split text in paragraphs
        text = text.split("\n\n")

        # using the helper function "remove_newline" to eliminate "\n" characters from the text
        text = remove_newline(text)

        return text

    def chunkEmbedIndex(self, file_path, sentence_limit, chunk_limit, min_characters):
        text = self.__preProcessInput(file_path)
        
        # Load pretrained English Language Model to separate the text into sentences
        nlp = spacy.load('en_core_web_sm') 

        chunks = []
        sentences = []

        # generate doc pipeline with nlp
        # allows to process the data as a stream and buffer the paragraphs in batches instead of one by one
        doc_pipeline = nlp.pipe(text, batch_size = 5, n_process = 1)

        # split the doc into sentences and create chunks that contain n = "sentence_limit" sentences
        # once m = "chunk_limit" chunks have been collected, create chunk embeddings and index
        for doc in tqdm(doc_pipeline):
            for sent in doc.sents:
                sentences.append(sent.text)

                if len(sentences) >= sentence_limit:
                    chunk = " ".join(sentences)

                    # if the number of characters in the current chunk is less than the minimum required, 
                    # then add another sentence and count again before embedding the text
                    if len(chunk) < min_characters:
                        continue

                    # once the chunk has the minimum length, append it to chunks
                    chunks.append(chunk)
                    # remove the first sentence and keep the other two to overlap with the following sentence
                    sentences = sentences[1:]

                if len(chunks) == chunk_limit:
                    # embed and index
                    embed_index_text(chunks, self.client)

                    # clear list of chunks
                    chunks = []

        # if there are sentences/chunks that haven't been embedded and indexed, do so
        if len(sentences) != 0:
            # append sentences to remaining chunks
            chunks.append(" ".join(sentences))

            embed_index_text(chunks, self.client)

            sentences = []
            chunks = []




In [4]:
model = FlagModel('BAAI/bge-small-zh-v1.5', use_fp16 = True)

In [5]:
# instantiate Python client for Elasticsearch
client = Elasticsearch("http://elasticsearch:9200")

In [6]:
client.indices.exists(index = "les_miserables_index")

HeadApiResponse(True)

In [7]:
# create index
createIndex(client, "les_miserables_index")

In [8]:
# instantiate IndexText class
les_miserables = IndexText(model, client)

In [28]:
# text = les_miserables.preProcessInput("../data/LesMiserables.txt")

In [10]:
# define before calling the chunkEmbedIndex method in the IndexText class
max_sentence = 6
max_chunk = 64
min_characters = 128

In [ ]:
# index the text
les_miserables.chunkEmbedIndex("../data/LesMiserables.txt", max_sentence, max_chunk, min_characters)

31it [00:00, 64.56it/s]You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
343it [04:36,  1.24it/s]
